# Large results: write a file, don't inline a table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/12_large_data.ipynb)

`add_features` puts every row into the widget's state as JSON. That is the right thing for a few thousand peaks, and the wrong thing by a hundred thousand — the whole table has to be serialized, pushed through the notebook's comm channel, and held in the browser's memory whether or not you ever look at it.

The alternative is what genome browsers have always done: write a **real indexed file** and read it *by byte range*. `add_local_file` pushes one from this kernel into the browser, where JBrowse seeks into it through its index exactly as it would a file on a web server — but with no server, no CORS, and no public bucket. Only the bytes for the region on screen are ever touched.

We'll build a real one: every **NCBI RefSeq exon in the human genome**, straight from UCSC.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy pysam pyBigWig

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The analysis

One download of the RefSeq transcript table, exploded into exons. This is ordinary pandas — nothing here knows about JBrowse yet.

In [ ]:
import numpy as np
import pandas as pd

COLS = ("bin name chrom strand txStart txEnd cdsStart cdsEnd exonCount "
        "exonStarts exonEnds score name2 cdsStartStat cdsEndStat exonFrames").split()
tx = pd.read_csv(
    "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/ncbiRefSeq.txt.gz",
    sep="\t",
    names=COLS,
)
tx = tx[tx.chrom.str.match(r"^chr(\d+|X|Y)$")]

# one row per exon
exons = pd.DataFrame(
    {
        "chrom": tx.chrom.str.removeprefix("chr").repeat(tx.exonCount).values,
        "start": np.concatenate(
            [np.fromstring(s.rstrip(","), sep=",", dtype=np.int64) for s in tx.exonStarts]
        ),
        "end": np.concatenate(
            [np.fromstring(e.rstrip(","), sep=",", dtype=np.int64) for e in tx.exonEnds]
        ),
        "name": tx.name2.repeat(tx.exonCount).values,
    }
).sort_values(["chrom", "start"], kind="stable")

print(f"{len(exons):,} exons")

## What inlining would cost

`features_track` builds the config `add_features` would send, so we can price a row before committing to two million of them.

In [ ]:
import json

from jbrowse_anywidget import features_track

sample = features_track(exons.head(20_000).to_dict("records"), name="sample")
per_row = len(json.dumps(sample)) / 20_000
print(f"inlined: ~{per_row * len(exons) / 1e6:,.0f} MB of JSON")

Around **200 MB** — through a websocket, into browser memory, to draw a few hundred exons at a time. Now the same data as a file.

## As a tabix file

BED, bgzipped and indexed — the same pair of files you'd host on a server. `pysam` writes both.

`add_local_file` registers the bytes under the file's name and picks up the `.tbi` sibling automatically. After that the name **is** the URL: `add_track` infers `BedTabixAdapter` from the `.bed.gz` extension and finds the index by name, exactly as it would for a remote file.

In [ ]:
import os

import pysam

exons.to_csv("exons.bed", sep="\t", header=False, index=False)
pysam.tabix_compress("exons.bed", "exons.bed.gz", force=True)
pysam.tabix_index("exons.bed.gz", preset="bed", force=True)

size = (os.path.getsize("exons.bed.gz") + os.path.getsize("exons.bed.gz.tbi")) / 1e6
print(f"tabix: {size:.1f} MB, and the view reads only the part it shows")

In [ ]:
from jbrowse_anywidget import LinearGenomeView

view = LinearGenomeView(assembly="hg38", location="17:7,668,400..7,687,500")
view.add_track(view.add_local_file("exons.bed.gz"))
view

That's TP53, drawn from a two-million-feature file that never left this kernel. Pan or zoom and the view fetches the next slice through the tabix index — the cost of moving is the same as it would be for a file on a server.

## As a bigWig

For a quantitative signal, bigWig is the better container: it stores **precomputed zoom levels**, so viewing a whole chromosome reads a summary rather than every underlying point. Here that's exon density per 100 kb — a crude gene-density map of the genome.

In [ ]:
import pyBigWig

BIN = 100_000
chrom_len = exons.groupby("chrom").end.max()
order = [str(c) for c in range(1, 23)] + ["X", "Y"]
order = [c for c in order if c in chrom_len.index]

bw = pyBigWig.open("exon_density.bw", "w")
bw.addHeader([(c, int(chrom_len[c]) + BIN) for c in order])
for c in order:
    binned = (exons.loc[exons.chrom == c, 'start'] // BIN * BIN).value_counts().sort_index()
    bw.addEntries(
        [c] * len(binned),
        binned.index.astype(int).tolist(),
        ends=(binned.index + BIN).astype(int).tolist(),
        values=binned.values.astype(float).tolist(),
    )
bw.close()
size = os.path.getsize("exon_density.bw") / 1e6
print(f"bigWig: {size:.1f} MB, with zoom levels baked in")

In [ ]:
view.add_track(view.add_local_file("exon_density.bw"))
view.location = "17"
view

## Which to use

| | `add_features` | `add_local_file` |
|---|---|---|
| data | a DataFrame or list of dicts | a real file you wrote |
| cost | whole table as JSON, always resident | bytes for the visible region |
| good to | a few thousand rows | as large as you like |
| formats | features only | anything JBrowse reads |

`add_local_file` is not limited to the two formats above — a sorted+indexed BAM or CRAM, a bgzipped VCF, a `.hic`, a bigBed all work the same way, because the browser is opening them with the same adapters it uses for remote files. Register the index under its conventional sibling name (`reads.bam` + `reads.bam.bai`) and the adapter finds it.

The files here are written to the notebook's working directory; nothing keeps them afterwards, and nothing was uploaded anywhere.